<a href="https://colab.research.google.com/github/Naman27-11/Event-Ticket-Booking-and-Management/blob/main/SourceCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import uuid

class Event:
    """Manages structural data parameters for individual events."""
    def __init__(self, event_id: str, name: str, total_seats: int, ticket_price: float):
        self.event_id = event_id
        self.name = name
        self.total_seats = total_seats
        self.available_seats = total_seats
        self.ticket_price = ticket_price

    def reserve_seats(self, count: int) -> bool:
        if count <= 0 or self.available_seats < count:
            return False
        self.available_seats -= count
        return True

    def release_seats(self, count: int):
        if count > 0 and (self.available_seats + count) <= self.total_seats:
            self.available_seats += count


class Booking:
    """Represents a transaction state structure for a user booking."""
    def __init__(self, booking_id: str, event_id: str, customer_name: str, seats_booked: int, total_cost: float):
        self.booking_id = booking_id
        self.event_id = event_id
        self.customer_name = customer_name
        self.seats_booked = seats_booked
        self.total_cost = total_cost


class TicketBookingSystem:
    """Core engine containing modules and handling core state workflows."""
    def __init__(self):
        # Using dynamic dictionary lookups to achieve both performance and scalability requirements
        self.events = {}
        self.bookings = {}

    # MODULE 1: Admin Module Functions
    def admin_create_event(self, event_id: str, name: str, total_seats: int, ticket_price: float) -> str:
        if not event_id or not name:
            raise ValueError("Event ID and Name cannot be empty values.")
        if total_seats <= 0 or ticket_price <= 0:
            raise ValueError("Seats count and ticket prices must be positive numbers.")
        if event_id in self.events:
            raise ValueError(f"An event with ID '{event_id}' already exists.")

        new_event = Event(event_id, name, total_seats, ticket_price)
        self.events[event_id] = new_event
        return f"Success: Event '{name}' created successfully with ID [{event_id}]."

    # MODULE 2: User Search & Booking Module Functions
    def get_all_events(self) -> list:
        return list(self.events.values())

    def user_book_ticket(self, event_id: str, customer_name: str, seats_requested: int) -> str:
        # FIXED: Removed the accidental 'biographies' typo here
        if event_id not in self.events:
            raise KeyError(f"Requested Event ID '{event_id}' does not exist.")
        if not customer_name.strip():
            raise ValueError("Customer name cannot be empty.")
        if seats_requested <= 0:
            raise ValueError("You must book at least 1 or more seats.")

        event = self.events[event_id]
        if not event.reserve_seats(seats_requested):
            raise OverflowError(f"Booking Failed: Only {event.available_seats} seats are left.")

        booking_id = str(uuid.uuid4())[:8].upper()  # Generates short human-readable unique reference
        total_cost = seats_requested * event.ticket_price

        new_booking = Booking(booking_id, event_id, customer_name, seats_requested, total_cost)
        self.bookings[booking_id] = new_booking

        return f"Booking Confirmed! ID: {booking_id} | Total Cost: ${total_cost:.2f}"

    # MODULE 3: Cancellation & Status Module Functions
    def user_cancel_ticket(self, booking_id: str) -> str:
        if booking_id not in self.bookings:
            raise KeyError(f"Transaction ID '{booking_id}' not found in records.")

        booking = self.bookings[booking_id]
        event = self.events[booking.event_id]

        # Release the locked seat allocations back to the main event pool
        event.release_seats(booking.seats_booked)
        del self.bookings[booking_id]

        return f"Success: Booking Reference [{booking_id}] cancelled. Refund of ${booking.total_cost:.2f} processed."


# MAIN CONSOLE EXECUTION FLOW (Logical Workflow Environment)
def main():
    system = TicketBookingSystem()

    # Pre-populating a few records for structural context and evaluation testing
    system.admin_create_event("E101", "Music Festival", 100, 75.0)
    system.admin_create_event("E102", "Tech Conference", 50, 120.0)

    print("==================================================")
    print("WELCOME TO THE CONSOLE EVENT TICKET ENGINE")
    print("==================================================")

    while True:
        print("\n--- GLOBAL OPTIONS ---")
        print("1. Admin Module: Add New Event")
        print("2. User Module: View Events & Book Tickets")
        print("3. User Module: Cancel Booked Ticket")
        print("4. Exit System Application")

        try:
            choice = input("Select an option (1-4): ").strip()

            if choice == "1":
                print("\n[ADMIN MENU]")
                eid = input("Enter unique Event ID: ").strip()
                name = input("Enter Event Name: ").strip()
                seats = int(input("Enter Maximum Seating Capacity: "))
                price = float(input("Enter Base Ticket Price ($): "))

                result = system.admin_create_event(eid, name, seats, price)
                print(result)

            elif choice == "2":
                print("\n[AVAILABLE EVENTS COLLECTION]")
                events_list = system.get_all_events()
                if not events_list:
                    print("No active events currently scheduled.")
                    continue

                print(f"{'ID':<10} | {'Event Name':<20} | {'Available Seats':<15} | {'Price':<8}")
                print("-" * 60)
                for e in events_list:
                    print(f"{e.event_id:<10} | {e.name:<20} | {e.available_seats:<15} | ${e.ticket_price:<8.2f}")

                print("\n[BOOKING REQUEST INTERFACE]")
                target_id = input("Enter Event ID you want to join: ").strip()
                cust_name = input("Enter your Full Name: ").strip()
                qty = int(input("Enter number of seats to buy: "))

                invoice_msg = system.user_book_ticket(target_id, cust_name, qty)
                print(invoice_msg)

            elif choice == "3":
                print("\n[TICKET CANCELLATION ENGINE]")
                target_booking_id = input("Enter your 8-digit unique Booking ID: ").strip()
                cancel_msg = system.user_cancel_ticket(target_booking_id)
                print(cancel_msg)

            elif choice == "4":
                print("\nThank you for using the Ticket Engine. Program Terminated Safely.")
                break
            else:
                print("Runtime Error: Invalid operational choice selection. Try numbers 1 to 4.")

        except ValueError as ve:
            print(f"Input Format Error: {ve}. Please pass correct numeric fields.")
        except (KeyError, OverflowError) as system_err:
            print(f"Business Rule Execution Blocked: {system_err}")
        except Exception as general_err:
            print(f"Critical Trapped Error: {general_err}")


if __name__ == "__main__":
    main()


WELCOME TO THE CONSOLE EVENT TICKET ENGINE

--- GLOBAL OPTIONS ---
1. Admin Module: Add New Event
2. User Module: View Events & Book Tickets
3. User Module: Cancel Booked Ticket
4. Exit System Application
Select an option (1-4): 2

[AVAILABLE EVENTS COLLECTION]
ID         | Event Name           | Available Seats | Price   
------------------------------------------------------------
E101       | Music Festival       | 100             | $75.00   
E102       | Tech Conference      | 50              | $120.00  

[BOOKING REQUEST INTERFACE]
Enter Event ID you want to join: E101
Enter your Full Name: Naman
Enter number of seats to buy: 1
Booking Confirmed! ID: EEF7C5EB | Total Cost: $75.00

--- GLOBAL OPTIONS ---
1. Admin Module: Add New Event
2. User Module: View Events & Book Tickets
3. User Module: Cancel Booked Ticket
4. Exit System Application
Select an option (1-4): 3

[TICKET CANCELLATION ENGINE]
Enter your 8-digit unique Booking ID: EEF7C5EB
Success: Booking Reference [EEF7C5EB] can